# Light Rider Quantum Quickstart

Build a quantum circuit with the [`lightrider`](https://pypi.org/project/lightrider/) Python SDK, check it on a free local simulator, then submit it to the IonQ Forte-1 through the Light Rider platform.

You need a Light Rider API key (`lr_...`). Create one at [platform.lightriderinc.com/settings/keys](https://platform.lightriderinc.com/settings/keys).

In [ ]:
import sys

major, minor = sys.version_info[:2]
print(f"Python {major}.{minor} detected.")
if major != 3 or minor < 10:
    print("\u26a0\ufe0f This notebook requires Python 3.10 or above. In Colab: Runtime \u2192 Change runtime type.")

Your API key comes from [platform.lightriderinc.com/settings/keys](https://platform.lightriderinc.com/settings/keys),
the same site this notebook submits jobs to. Paste it into the cell below.


In [ ]:
%pip install -q uv
!uv pip install -q --system lightrider==1.5.2 requests

import requests

#@markdown Paste your Light Rider API key below, then run this cell.
api_key = "" #@param {type:"string"}
base_url = "https://platform.lightriderinc.com"

api_key = api_key.strip()
if not api_key.startswith("lr_"):
    raise ValueError(
        f"Expected a Light Rider API key beginning with 'lr_'. "
        f"Create one at {base_url}/settings/keys."
    )

session = requests.Session()
session.headers["Authorization"] = f"Bearer {api_key}"

## Build a circuit with the SDK

A Bell pair: Hadamard on qubit 0, then CNOT onto qubit 1, then measure both qubits.

In [ ]:
from lightrider import Circuit

circuit = Circuit(2, name="colab-bell")
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()

print(circuit)

## Dry-run locally (free)

The SDK ships local simulators, so you can verify the circuit's physics before spending any credits. A Bell pair should give only `00` and `11`, roughly 50/50.

In [ ]:
from lightrider import get_backend

local = get_backend("statevector")
print("local counts:", local.run(circuit, shots=1000).result().get_counts())

## Pick a platform backend

Valid `backend` values for submission:

- **Mock (free, unlimited):** `iqm-garnet-mock`, `iqm-emerald-mock`, `iqm-sirius-mock`, `rigetti-cepheus-mock`, `ionq-simulator`, `ionq-forte-1-simulator`
- **Real QPU (costs credits, requires a prior purchase):** `iqm-garnet`, `iqm-emerald`, `iqm-sirius`, `rigetti-cepheus`
- **Real QPU priced per job (costs credits, get the price first):** `ionq-forte-1`, `ionq-forte-enterprise-1`. Their own notebooks include the price step.

## Get the price

IonQ Forte-1 is priced per job, not per shot. This cell asks the platform for the price of this exact circuit and shot count, in credits. Nothing is charged here.

In [ ]:
backend = "ionq-forte-1"  # change this to the backend you want to use
shots = 1000

estimate = session.post(
    f"{base_url}/api/lr/quantum/estimate",
    json={"backend": backend, "circuit": circuit.to_payload(), "shots": shots},
)
if not estimate.ok:
    detail = estimate.json()
    raise RuntimeError(f"{detail.get('error')}: {detail.get('message', '')}")
quote = estimate.json()
price_credits = quote["priceCredits"]

print(f"Price for this job: {price_credits:,} credits")
if quote.get("minimumCredits"):
    print(f"Minimum per job: {quote['minimumCredits']:,} credits")
print(f"Your balance: {quote['balanceCredits']:,} credits")

## Submit with the confirmed price

Run the next cell only if you accept the price above. It charges that many credits and sends the job to the real QPU. If the price has gone up since the quote, the job is refused and nothing is charged; run the price cell again.

In [ ]:
response = session.post(
    f"{base_url}/api/lr/quantum/submit",
    json={
        "backend": backend,
        "circuit": circuit.to_payload(),
        "shots": shots,
        "confirmedPriceCredits": price_credits,
    },
)
if not response.ok:
    # 402: no purchase yet or not enough credits. 409: price missing or changed.
    # 503: the system is unavailable (credits are returned).
    detail = response.json()
    raise RuntimeError(f"{detail.get('error')}: {detail.get('message', '')}")
job = response.json()
print("Job submitted:", job["job_uuid"])
print("Status:", job["status"])
print("Charged:", job["priceCredits"], "credits")

In [ ]:
import time

job_id = job["job_uuid"]

# Real QPU jobs wait in IonQ's queue, which can take hours. If this cell
# stops before the job finishes, run it again later; the job keeps running.
for _ in range(60):
    status_response = session.get(f"{base_url}/api/lr/quantum/jobs/{job_id}")
    status_response.raise_for_status()
    status_data = status_response.json()
    print("Status:", status_data["status"])

    if status_data.get("isInTerminalState"):
        break
    time.sleep(10)
else:
    raise TimeoutError("The job is still queued. Run this cell again later, or check platform.lightriderinc.com/jobs.")

result_response = session.get(f"{base_url}/api/lr/quantum/jobs/{job_id}/result")
result_response.raise_for_status()
counts = result_response.json()["counts"]
print("Counts:", counts)

In [ ]:
import matplotlib.pyplot as plt

plt.bar(counts.keys(), counts.values())
plt.xlabel("Measurement outcome")
plt.ylabel("Count")
plt.title(f"Bell pair on {backend}")
plt.show()

This circuit creates a **Bell pair**: two qubits entangled so they always agree when measured. Real hardware has some noise, so most results are `00` and `11`, roughly 50/50, with a few `01` and `10`. That's normal noise, not a broken circuit.

## Billing and plans

IonQ Forte-1 is priced per job, not per shot. The price cell shows the cost of this exact job in credits before anything is charged. Submitting with `confirmedPriceCredits` set to that price charges it and runs the job. If you submit without it, or the price has gone up since the quote, you get a 409 (`price_confirmation_required` or `price_changed`) and nothing is charged.

Real QPUs require having purchased credits at least once (the free signup credits alone don't unlock real hardware). If you haven't purchased yet, you'll get a 402 (`purchase_required`). If your balance is lower than the price, you'll get a 402 (`insufficient_credits`). If the system is unavailable, for example during maintenance, you'll get a 503 and your credits are returned. Buy credits at [/settings/manage-credits/purchase](https://platform.lightriderinc.com/settings/manage-credits/purchase).

Your submitted jobs, from this notebook and from the web dashboard alike, are listed at [platform.lightriderinc.com/jobs](https://platform.lightriderinc.com/jobs).